In [35]:
from random import random, randint, seed, sample
from icecream import ic
from math import sqrt

In [36]:
from upq import PriorityQueue

In [ ]:

class KlotskiProblem:
  """Contiene tutti i dati STATICI del problema"""

  def __init__(self, rows, cols, exit_cols, target, raw_grid):
    self.rows = rows
    self.cols = cols
    self.exit_cols = exit_cols 
    self.target = target 

    self.pieces, self.shapes, initial_coords = self._parse_grid(raw_grid)

    self.target_idx = self.pieces.index(self.target)
    target_h, target_w = self.shapes[self.target]
    exit_cols = sorted(set(self.exit_cols))

    if len(exit_cols) == 1:
      goal_cols = exit_cols
    else:
      if exit_cols != list(range(exit_cols[0], exit_cols[-1] + 1)):
        raise ValueError("Le colonne dell'uscita devono essere consecutive.")
      goal_cols = range(exit_cols[-1] - target_w + 1, exit_cols[0] + 1)
      if not goal_cols:
        raise ValueError("Il pezzo target è più stretto dell'uscita.")

    self.goal_positions = tuple((rows - target_h, col) for col in goal_cols)

    self.initial_state = State(initial_coords)

  def _parse_grid(self, grid):
    """Scansiona la matrice iniziale per trovare angolo in alto a sinistra e dimensioni (h, w)."""
    if len(grid) != self.rows or any(len(row) != self.cols for row in grid):
      row_lengths = [len(row) for row in grid]
      raise ValueError(
        f"Griglia non compatibile: attese {self.rows} righe e {self.cols} colonne, "
        f"ricevute {len(grid)} righe con lunghezze {row_lengths}."
      )

    found_cells = {}
    for r in range(self.rows):
      for c in range(self.cols):
        token = grid[r][c]
        if token != '.':
          if token not in found_cells:
            found_cells[token] = []
          found_cells[token].append((r, c))

    pieces = tuple(sorted(found_cells.keys()))
    shapes = {}
    initial_coords = []

    for p in pieces:
      cells = found_cells[p]
      min_r = min(r for r, c in cells)
      max_r = max(r for r, c in cells)
      min_c = min(c for r, c in cells)
      max_c = max(c for r, c in cells)

      h = max_r - min_r + 1
      w = max_c - min_c + 1
      shapes[p] = (h, w)

      initial_coords.append((min_r, min_c))

    return pieces, shapes, tuple(initial_coords)

In [ ]:
class State:
  """Contiene i dati DINAMICI: solo la tupla di coordinate ((r1, c1), (r2, c2), ...)"""

  def __init__(self, coords):
    # coords è una tupla di tuple: ((0, 1), (0, 2), ...)
    self.coords = coords

  def to_grid(self, problem):
    """Ricostruisce la matrice RxC a partire dalle coordinate"""
    grid = [['.' for _ in range(problem.cols)] for _ in range(problem.rows)]
    for i, (r, c) in enumerate(self.coords):
      piece_name = problem.pieces[i]
      h, w = problem.shapes[piece_name]
      for dr in range(h):
        for dc in range(w):
          grid[r + dr][c + dc] = piece_name
    return grid


In [39]:
def is_goal(state, problem):
  """Controlla se lo stato è uno stato obiettivo."""
  grid = state.to_grid(problem)
  return all(grid[problem.rows - 1][col] == problem.target for col in problem.exit_cols)

In [ ]:
def get_neighbors(state, problem):
  """Genera tutti gli stati raggiungibili da quello corrente."""
  neighbors = []
  grid = state.to_grid(problem)

  for i, (r, c) in enumerate(state.coords):
    piece_name = problem.pieces[i]
    h, w = problem.shapes[piece_name]

    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
      new_r = r + dr
      new_c = c + dc

      if new_r < 0 or new_c < 0 or new_r + h > problem.rows or new_c + w > problem.cols:
        continue

      can_move = True
      for dh in range(h):
        for dw in range(w):
          if grid[new_r + dh][new_c + dw] != '.' and grid[new_r + dh][new_c + dw] != piece_name:
            can_move = False
            break
        if not can_move:
          break

      if can_move:
        new_coords = list(state.coords)
        new_coords[i] = (new_r, new_c)
        neighbors.append(State(tuple(new_coords)))

  return neighbors

In [ ]:
def heuristic(state, problem):
  """Calcola la distanza di Manhattan tra il pezzo target e l'uscita"""
  target_r, target_c = state.coords[problem.target_idx]
  return min(
    abs(target_r - goal_r) + abs(target_c - goal_c)
    for goal_r, goal_c in problem.goal_positions
  )

In [ ]:
def a_star(problem):
  """Implementa l'algoritmo A* per risolvere il problema di Klotski"""
  initial_state = problem.initial_state
  frontier = PriorityQueue()
  frontier.g_score[initial_state] = 0
  frontier.parent[initial_state] = None
  frontier.push(initial_state, heuristic(initial_state, problem))

  while not frontier.is_empty():
    current = frontier.pop()

    if current is None:
      break

    if is_goal(current, problem):
      path = []
      while current is not None:
        path.append(current)
        current = frontier.parent[current]
      path.reverse()
      return path

    for neighbor in get_neighbors(current, problem):

      if neighbor in frontier.visited:
        continue

      new_cost = frontier.g_score[current] + 1
      if neighbor not in frontier.g_score or new_cost < frontier.g_score[neighbor]:
        frontier.g_score[neighbor] = new_cost
        priority = new_cost + heuristic(neighbor, problem)
        frontier.push(neighbor, priority)
        frontier.parent[neighbor] = current

  return None

In [65]:
# parametri del problema

target = 'A'
exit_cols = [1, 2]
rows = 4
cols = 4
grid = """
B 2 . 3
B 1 A A
4 . A A
C C D D
"""

raw_grid = [line.split() for line in grid.strip().splitlines()]

### A*

In [68]:
solution = a_star(KlotskiProblem(rows, cols, exit_cols, target, raw_grid))

In [67]:
for state in solution:
    grid = state.to_grid(KlotskiProblem(rows, cols, exit_cols, target, raw_grid))
    for row in grid:
        print(' '.join(row))
    print()

TypeError: 'NoneType' object is not iterable

In [64]:
print(f"Numero di mosse: {len(solution) - 1}")

Numero di mosse: 179


### DFS

In [ ]:
def DFS(problem):
  """Implementa l'algoritmo Depth-First Search per risolvere il problema di Klotski"""
  initial_state = problem.initial_state
  stack = [initial_state]
  visited = set()
  parent = {initial_state: None}

  while stack:
    current = stack.pop()

    if current in visited:
      continue

    visited.add(current)

    if is_goal(current, problem):
      path = []
      while current is not None:
        path.append(current)
        current = parent[current]
      path.reverse()
      return path

    for neighbor in get_neighbors(current, problem):
      if neighbor not in visited:
        stack.append(neighbor)
        parent[neighbor] = current

  return None

In [ ]:
solution = DFS(KlotskiProblem(rows, cols, exit_cols, target, raw_grid))

In [ ]:
for state in solution:
    grid = state.to_grid(KlotskiProblem(rows, cols, exit_cols, target, raw_grid))
    for row in grid:
        print(' '.join(row))
    print()

In [ ]:
print(f"Numero di mosse: {len(solution) - 1}")

### BFS

In [ ]:
def BFS(problem):
  """Implementa l'algoritmo di ricerca in ampiezza (BFS) per risolvere il problema di Klotski"""
  initial_state = problem.initial_state
  frontier = PriorityQueue()
  frontier.g_score[initial_state] = 0
  frontier.parent[initial_state] = None
  frontier.push(initial_state, 0)

  while not frontier.is_empty():
    current = frontier.pop()

    if current is None:
      break

    if is_goal(current, problem):
      path = []
      while current is not None:
        path.append(current)
        current = frontier.parent[current]
      path.reverse()
      return path

    for neighbor in get_neighbors(current, problem):

      if neighbor in frontier.visited:
        continue

      new_cost = frontier.g_score[current] + 1
      if neighbor not in frontier.g_score or new_cost < frontier.g_score[neighbor]:
        frontier.g_score[neighbor] = new_cost
        priority = new_cost
        frontier.push(neighbor, priority)
        frontier.parent[neighbor] = current

  return None

In [ ]:
solution = BFS(KlotskiProblem(rows, cols, exit_cols, target, raw_grid))

In [ ]:
for state in solution:
    grid = state.to_grid(KlotskiProblem(rows, cols, exit_cols, target, raw_grid))
    for row in grid:
        print(' '.join(row))
    print()

In [ ]:
print(f"Numero di mosse: {len(solution) - 1}")